# DAVID-Net Training — Kaggle

**Datasets:**
- **Training:** FakeAVCeleb (quadrant labels) + LAV-DF (localization labels)
- **Cross-dataset eval:** DFDC-10, DeepFakeTIMIT, Celeb-DF v2
- **Audio eval:** ASVspoof-2019, In-the-Wild, WaveFake

**Pipeline:**
1. Setup → 2. Discover → 3. Extract → 4. Build manifests → 5. Precompute features
6. QACP Stage 0 → 7. Stage 1 (3 seeds) → 8. Cross-dataset eval

**Architecture compliance:** VideoMAE + WavLM, cosine schedule, gradient checkpointing,
generator-balanced sampling, augmentation, focal loss, MI penalty.

In [ ]:
# Cell 1: GPU check + install deps
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU.")
!pip install -q transformers accelerate scikit-learn jiwer datasets

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

In [ ]:
# Cell 4: Discover + Auto-download all 8 datasets
# If datasets are mounted via UI, uses them directly.
# If not mounted, downloads them automatically via Kaggle API.
import subprocess, zipfile
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
DOWNLOAD_DIR = WORKING / "kaggle_datasets"
DOWNLOAD_DIR.mkdir(exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg"}

def has_media_files(d):
    for f in d.rglob("*"):
        if f.suffix.lower() in VIDEO_EXTS | AUDIO_EXTS:
            return True
    return False

# All 8 datasets: slug -> (friendly name, mount paths to check)
DATASETS = {
    "fakeavceleb": {
        "slug": "aicontentdetections/fakeavceleb-v1-2",
        "mounts": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    },
    "lav-df": {
        "slug": "aicontentdetections/lav-df",
        "mounts": ["aicontentdetections/lav-df"],
    },
    "dfdc-10": {
        "slug": "pranay22077/dfdc-10",
        "mounts": ["pranay22077/dfdc-10"],
    },
    "deepfaketimit": {
        "slug": "fahimaislam1812/deepfaketimit",
        "mounts": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    },
    "celeb-df-v2": {
        "slug": "reubensuju/celeb-df-v2",
        "mounts": ["reubensuju/celeb-df-v2"],
    },
    "asvpoof-2019": {
        "slug": "anishsarkar22/asvpoof-2019-dataset-la",
        "mounts": ["anishsarkar22/asvpoof-2019-dataset-la"],
    },
    "in-the-wild": {
        "slug": "abdallamohamed312/in-the-wild-audio-deepfake",
        "mounts": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    },
    "wavefake": {
        "slug": "walimuhammadahmad/fakeaudio",
        "mounts": ["walimuhammadahmad/fakeaudio", "andreadiubaldo/wavefake-test"],
    },
}

def find_mounted(slug_paths):
    """Check if dataset is already mounted."""
    for p in slug_paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                return candidate
    return None

def download_dataset(slug, friendly):
    """Download dataset from Kaggle API and extract."""
    dst = DOWNLOAD_DIR / friendly
    if dst.exists() and any(dst.rglob("*.mp4")) or any(dst.rglob("*.wav")) or any(dst.rglob("*.flac")):
        print(f"  {friendly}: already downloaded")
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    print(f"  {friendly}: downloading from {slug}...", end=" ")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", str(dst), "--unzip"],
            capture_output=True, text=True, timeout=3600
        )
        if result.returncode == 0:
            print("OK")
            return dst
        else:
            print(f"FAIL: {result.stderr[:200]}")
            return None
    except subprocess.TimeoutExpired:
        print("TIMEOUT (1h limit)")
        return None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

datasets = {}
for friendly, info in DATASETS.items():
    # 1. Check if mounted
    mounted = find_mounted(info["mounts"])
    if mounted:
        datasets[friendly] = mounted
        print(f"  {friendly} -> {mounted} (mounted)")
        continue
    # 2. Check if previously downloaded
    downloaded = DOWNLOAD_DIR / friendly
    if downloaded.exists() and has_media_files(downloaded):
        datasets[friendly] = downloaded
        print(f"  {friendly} -> {downloaded} (cached)")
        continue
    # 3. Download from Kaggle
    result = download_dataset(info["slug"], friendly)
    if result:
        datasets[friendly] = result

print(f"\nFound {len(datasets)}/8 datasets.")
if len(datasets) < 8:
    missing = set(DATASETS) - set(datasets)
    print(f"Missing: {missing}")
    print("Add via + Add Data or check Kaggle API credentials.")

In [ ]:
# Cell 5: Extract compressed datasets
import zipfile, tarfile, shutil

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg"}

def has_media_files(d):
    for f in d.rglob("*"):
        if f.suffix.lower() in VIDEO_EXTS | AUDIO_EXTS:
            return True
    return False

def find_multipart_zip_base(src):
    """Find base name of multi-part zip (e.g. LAV-DF.zip.001 -> LAV-DF.zip)."""
    parts = sorted(src.rglob("*.zip.*"))
    if not parts:
        return None
    # Get base name: LAV-DF.zip.001 -> LAV-DF.zip
    base = parts[0].name.rsplit(".", 1)[0]
    # Check if .001 exists (first part)
    first = parts[0].parent / (base + ".001")
    if first.exists():
        return parts[0].parent / base
    # Some use .zip.001 naming
    base2 = parts[0].name.split(".zip")[0] + ".zip"
    return parts[0].parent / base2

def extract_multipart_zip(name, base_path, dst):
    """Extract multi-part zip by opening .001 with zipfile (handles spanning)."""
    first_part = Path(str(base_path) + ".001")
    if not first_part.exists():
        print(f"  {name}: first part {first_part.name} not found")
        return False
    print(f"  {name}: extracting multi-part zip ({first_part.name} + parts)...")
    try:
        # zipfile can read spanning zips if opened from .001
        with zipfile.ZipFile(str(first_part)) as zf:
            zf.extractall(dst)
        print(f"  {name}: multi-part extraction OK")
        return True
    except Exception as e:
        print(f"  {name}: multi-part extraction FAILED: {e}")
        # Fallback: concatenate parts then extract
        try:
            print(f"  {name}: trying concat fallback...")
            parts = sorted(first_part.parent.glob(base_path.name + ".*"))
            combined = dst / (base_path.name + ".combined.zip")
            with open(combined, "wb") as out:
                for p in parts:
                    with open(p, "rb") as inp:
                        shutil.copyfileobj(inp, out)
            with zipfile.ZipFile(str(combined)) as zf:
                zf.extractall(dst)
            combined.unlink()  # free space
            print(f"  {name}: concat fallback OK")
            return True
        except Exception as e2:
            print(f"  {name}: concat fallback FAILED: {e2}")
            return False

def extract_archives(name, src, dst):
    extracted_any = False
    # Check for multi-part zips first
    multipart_base = find_multipart_zip_base(src)
    if multipart_base:
        extracted_any = extract_multipart_zip(name, multipart_base, dst)
    # Regular zips
    for arch in src.rglob("*.zip"):
        if ".zip." in arch.name:  # skip multi-part parts
            continue
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
            extracted_any = True
        except Exception as e:
            print(f"FAIL: {e}")
    for arch in src.rglob("*.tar.gz"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with tarfile.open(arch, "r:gz") as tf:
                tf.extractall(dst)
            print("OK")
            extracted_any = True
        except Exception as e:
            print(f"FAIL: {e}")
    return extracted_any

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already extracted")
        continue
    if has_media_files(path):
        print(f"  {name}: loose media files found")
        marker.touch()
        continue
    print(f"  {name}: searching for archives...")
    extract_archives(name, path, dst)
    if has_media_files(dst):
        print(f"  {name}: extracted media found in {dst}")
    marker.touch()

print("\nExtraction done.")

In [ ]:
# Cell 6: Build ALL manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# === FakeAVCeleb (existing converter) ===
fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    print(f"FakeAVCeleb: {fakeav_root}")
    !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# === All other datasets (unified converter) ===
CONVERTERS = [
    ("lav-df", "lav-df"),
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    # For extracted datasets, also check DATA_DIR
    extracted_root = DATA_DIR / ds_name
    if not root or not root.exists():
        if extracted_root.exists() and has_media_files(extracted_root):
            root = extracted_root
            print(f"  {ds_name}: using extracted path {root}")
    if root and root.exists():
        print(f"\n--- {ds_name} ---")
        !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}
    else:
        print(f"  {ds_name}: NOT FOUND")

# === Summary ===
print("\n" + "="*50)
print("ALL MANIFESTS:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Precompute SSL features for training sets (FakeAVCeleb + LAV-DF)
import yaml

FAKEAV_MANIFEST = str(MANIFEST_DIR / "fakeavceleb.jsonl")
FAKEAV_ROOT = str(datasets.get("fakeavceleb", ""))
LAVDF_MANIFEST = str(MANIFEST_DIR / "lav-df.jsonl")
LAVDF_ROOT = str(DATA_DIR / "lav-df")
FEAT_CACHE = WORKING / "feature_cache"
FEAT_CACHE.mkdir(exist_ok=True)

FEAT_CFG = {
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 12, "freeze_feature_extractor": True,
    "d_model": 768, "n_frames": 16, "audio_len": 64000,
    "shard_root": None, "num_workers": 2,
}
feat_cfg_path = WORKING / "feat_cfg.yaml"
with open(feat_cfg_path, "w") as f:
    yaml.dump(FEAT_CFG, f)

# Extract features for both training datasets
TRAINING_DATASETS = [("FakeAVCeleb", FAKEAV_MANIFEST, FAKEAV_ROOT)]
lavdf_path = Path(LAVDF_MANIFEST)
if lavdf_path.exists():
    TRAINING_DATASETS.append(("LAV-DF", LAVDF_MANIFEST, LAVDF_ROOT))

for ds_name, manifest, root in TRAINING_DATASETS:
    n_cached = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
    print(f"\nPrecomputing SSL features for {ds_name}...")
    !cd {REPO} && python -m src.data.extract_features \
        --config {feat_cfg_path} \
        --manifest {manifest} \
        --out {FEAT_CACHE} \
        --batch-size 4 \
        --root-dir {root}
    n_after = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
    print(f"Cached features: {n_after} total clips (+{n_after - n_cached} from {ds_name})")

# Create combined training manifest (FakeAVCeleb + LAV-DF)
COMBINED_MANIFEST = WORKING / "train_combined.jsonl"
with open(COMBINED_MANIFEST, "w") as out:
    for manifest_path in [FAKEAV_MANIFEST, LAVDF_MANIFEST]:
        if Path(manifest_path).exists():
            with open(manifest_path) as f:
                for line in f:
                    out.write(line)
print(f"\nCombined training manifest: {COMBINED_MANIFEST}")
!wc -l {COMBINED_MANIFEST}

In [ ]:
# Cell 8: QACP Stage 0 config (NO feature_cache — needs raw data for transforms)
QACP_CONFIG = {
    "run_id": "qacp_stage0",
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 12, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None, "feature_cache": None,
    "train_manifest": str(COMBINED_MANIFEST),
    "root_dir": [FAKEAV_ROOT, LAVDF_ROOT],
    "modality_dropout": 0.0, "augment": False,
    "batch_size": 4, "num_workers": 2, "epochs": 20,
    "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
    "warmup_epochs": 2, "gradient_checkpointing": True,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
    "qacp_temperature": 0.1, "seed": 42,
}

qacp_path = WORKING / "qacp_config.yaml"
with open(qacp_path, "w") as f:
    yaml.dump(QACP_CONFIG, f)
print(f"QACP: decodes from MP4 (raw data for Griffin-Lim/self-blending)")

In [ ]:
# Cell 9: Run QACP Stage 0
!cd {REPO} && python -m src.training.pretrain_qacp \
    --config {qacp_path} \
    --run-id {QACP_CONFIG['run_id']}

In [ ]:
# Cell 10: Stage 1 config (init from QACP) — 3 seeds
import glob

qacp_ckpt = None
for p in sorted(glob.glob(str(WORKING / "runs/qacp_epoch*.pt")), reverse=True):
    qacp_ckpt = p
    break
if not qacp_ckpt:
    qacp_ckpt = str(WORKING / "runs/qacp_epoch19.pt")
print(f"QACP checkpoint: {qacp_ckpt}")

SEEDS = [42, 123, 456]
STAGE1_CONFIGS = []

for seed in SEEDS:
    cfg = {
        "run_id": f"stage1_seed{seed}",
        "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
        "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
        "video_backbone": "videomae", "audio_backbone": "wavlm",
        "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
        "freeze_blocks": 6, "freeze_feature_extractor": True,
        "init_from": qacp_ckpt,
        "n_frames": 16, "audio_len": 64000, "shard_root": None,
        "feature_cache": str(FEAT_CACHE),
        "train_manifest": str(COMBINED_MANIFEST),
        "val_manifest": str(SPLIT_DIR / "fakeavceleb" / "val.jsonl"),
        "root_dir": [FAKEAV_ROOT, LAVDF_ROOT],
        "modality_dropout": 0.15, "augment": True,
        "batch_size": 4, "num_workers": 2, "epochs": 30,
        "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
        "warmup_epochs": 2, "gradient_checkpointing": True,
        "log_every": 10,
        "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
        "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
        "seed": seed,
    }
    path = WORKING / f"stage1_seed{seed}_config.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    STAGE1_CONFIGS.append((seed, path, cfg))
    print(f"Seed {seed}: {cfg['run_id']}")

In [ ]:
# Cell 11: Run Stage 1 training (3 seeds)
for seed, config_path, cfg in STAGE1_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}...")
    print(f"{'='*60}")
    !cd {REPO} && python -m src.training.train \
        --config {config_path} \
        --run-id {cfg['run_id']}

In [ ]:
# Cell 11b: RESUME — pull checkpoints from HF and continue training
# Run this cell instead of Cell 11 if your session died mid-training.
import glob, torch
from src.utils.hf_backup import HFBackup

for seed, config_path, cfg in STAGE1_CONFIGS:
    run_id = cfg["run_id"]
    print(f"\n{'='*60}")
    print(f"Resuming {run_id} from HF...")
    print(f"{'='*60}")

    backup = HFBackup(run_id=run_id, local_dir=str(WORKING))

    # List what's on HF
    ckpts = backup.list_checkpoints()
    print(f"  Checkpoints on HF: {len(ckpts)}")
    for c in ckpts:
        print(f"    {c}")

    if not ckpts:
        print(f"  No checkpoints found — training from scratch")
        !cd {REPO} && python -m src.training.train \
            --config {config_path} \
            --run-id {run_id}
        continue

    # Find latest epoch number
    latest_ckpt = ckpts[-1]
    epoch_num = int(latest_ckpt.split("epoch_")[1].split(".pt")[0])
    total_epochs = cfg["epochs"]
    print(f"  Latest checkpoint: epoch {epoch_num}/{total_epochs}")

    if epoch_num >= total_epochs - 1:
        print(f"  Training already complete for {run_id}")
        continue

    # Download latest checkpoint
    local_path = backup.download_latest()
    if local_path:
        print(f"  Downloaded to: {local_path}")
        # Update config to resume from this checkpoint
        cfg["resume_from"] = local_path
        resume_cfg_path = WORKING / f"resume_{run_id}_config.yaml"
        with open(resume_cfg_path, "w") as f:
            yaml.dump(cfg, f)
        !cd {REPO} && python -m src.training.train \
            --config {resume_cfg_path} \
            --run-id {run_id} \
            --resume-from {local_path}
    else:
        print(f"  Download failed — training from scratch")
        !cd {REPO} && python -m src.training.train \
            --config {config_path} \
            --run-id {run_id}

In [ ]:
# Cell 12: Cross-dataset evaluation (ALL held-out datasets)
# Each eval is uploaded to HF immediately — survives session death.
# Re-run to skip already-completed evals.
import json

# All eval datasets (NOT used in training)
EVAL_DATASETS = {
    # Cross-dataset generalization (§2 Table 2)
    "dfdc-10": MANIFEST_DIR / "dfdc-10.jsonl",
    "deepfaketimit": MANIFEST_DIR / "deepfaketimit.jsonl",
    "celeb-df-v2": MANIFEST_DIR / "celeb-df-v2.jsonl",
    # Audio-only evaluation (§3)
    "asvpoof-2019": MANIFEST_DIR / "asvpoof-2019.jsonl",
    "in-the-wild": MANIFEST_DIR / "in-the-wild.jsonl",
    "wavefake": MANIFEST_DIR / "wavefake.jsonl",
}

# Download best checkpoints from HF if not local
from src.utils.hf_backup import HFBackup

all_results = {}
for seed, config_path, stage1_cfg in STAGE1_CONFIGS:
    run_id = stage1_cfg["run_id"]

    # Try local first, then HF
    best_ckpt = None
    for p in sorted(glob.glob(str(WORKING / f"runs/{run_id}_epoch*.pt")), reverse=True):
        best_ckpt = p
        break
    if not best_ckpt:
        print(f"Seed {seed}: no local checkpoint — downloading best from HF...")
        hf = HFBackup(run_id=run_id, local_dir=str(WORKING))
        best_ckpt = hf.download_best()
        if best_ckpt:
            print(f"  Downloaded: {best_ckpt}")
        else:
            print(f"  No checkpoint on HF either — skipping seed {seed}")
            continue

    seed_results = {}
    for ds_name, manifest in EVAL_DATASETS.items():
        if not manifest.exists():
            print(f"  {ds_name}: no manifest, skipping")
            continue
        print(f"\nSeed {seed} on {ds_name}...")
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate \
            --config {config_path} \
            --checkpoint {best_ckpt} \
            --manifest {manifest} \
            --out {report_path} \
            --run-id {run_id} \
            --ds-name {ds_name} \
            --skip-if-done
        if report_path.exists():
            with open(report_path) as f:
                r = json.load(f)
            seed_results[ds_name] = {
                "video_auc": r["video"]["auc"],
                "audio_auc": r["audio"]["auc"],
                "quadrant_acc": r["quadrant"]["acc"],
            }
            print(f"  v_auc={r['video']['auc']:.4f} a_auc={r['audio']['auc']:.4f}")
    all_results[f"seed{seed}"] = seed_results

# === Summary ===
print("\n" + "="*70)
print("CROSS-DATASET EVALUATION SUMMARY")
print("="*70)
print(f"{'Dataset':<20} {'Type':<15} {'V-AUC':<10} {'A-AUC':<10} {'Quad-Acc':<10}")
print("-"*70)
for seed_key, ds_results in all_results.items():
    for ds, m in ds_results.items():
        dtype = "AV" if ds in ["dfdc-10", "deepfaketimit", "celeb-df-v2"] else "Audio"
        print(f"{ds:<20} {dtype:<15} {m['video_auc']:<10.4f} {m['audio_auc']:<10.4f} {m['quadrant_acc']:<10.4f}")

# Save summary to HF
summary_path = WORKING / "eval_summary.json"
with open(summary_path, "w") as f:
    json.dump(all_results, f, indent=2)
# Upload summary for each seed
for seed, _, cfg in STAGE1_CONFIGS:
    try:
        hf = HFBackup(run_id=cfg["run_id"], local_dir=str(WORKING))
        api = hf._get_api()
        api.upload_file(
            path_or_fileobj=str(summary_path),
            path_in_repo=f"{hf.base_path}/eval/eval_summary.json",
            repo_id=hf.repo_id, repo_type=hf.repo_type,
        )
        break  # only need to upload once
    except Exception:
        pass

In [ ]:
# Cell 13: Create Model Card + Final HF Summary
# This uploads a README.md with all results to HF for easy access.
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
REPO_ID = "MoshinAli/david-net-av-backup"

# Build model card content
card = """# DAVID-Net — Audio-Visual Deepfake Detection

## Training Details
- **Backbone:** VideoMAE-Base + WavLM-Base+
- **Fusion:** Cross-modal transformer (self + cross attention)
- **Loss:** focal_bce_v + focal_bce_a + CE_quad + loc + sync + disentangle
- **Training:** FakeAVCeleb + LAV-DF, 3 seeds, 30 epochs each

## Cross-Dataset Evaluation Results

| Dataset | Type | V-AUC | A-AUC | Quad-Acc |
|---------|------|-------|-------|----------|
"""

for seed_key, ds_results in all_results.items():
    for ds, m in ds_results.items():
        dtype = "AV" if ds in ["dfdc-10", "deepfaketimit", "celeb-df-v2"] else "Audio"
        card += f"| {ds} | {dtype} | {m['video_auc']:.4f} | {m['audio_auc']:.4f} | {m['quadrant_acc']:.4f} |\n"

card += """
## Checkpoints
- `runs/<run_id>/best/best.pt` — Best model per seed
- `runs/<run_id>/checkpoints/epoch_NNNN.pt` — Per-epoch checkpoints
- `runs/<run_id>/eval/<ds_name>.json` — Per-dataset evaluation reports
- `runs/<run_id>/eval/eval_summary.json` — Combined evaluation summary

## Usage
```python
from huggingface_hub import hf_hub_download
path = hf_hub_download("MoshinAli/david-net-av-backup",
                       "runs/stage1_seed42/best/best.pt",
                       repo_type="model")
```
"""

# Upload model card
card_path = WORKING / "README.md"
with open(card_path, "w") as f:
    f.write(card)
api.upload_file(
    path_or_fileobj=str(card_path),
    path_in_repo="README.md",
    repo_id=REPO_ID, repo_type="model",
)
print("Model card uploaded to HF!")

# Final file listing
print("\n" + "="*50)
print("ALL FILES ON HF:")
for seed, _, cfg in STAGE1_CONFIGS:
    run = cfg["run_id"]
    try:
        files = list(api.list_repo_tree(REPO_ID,
                                         path_in_repo=f"runs/{run}",
                                         repo_type="model", recursive=True))
        print(f"\n{run}:")
        for f in files:
            if hasattr(f, 'path'): print(f"  {f.path}")
    except Exception as e:
        print(f"  {run}: {e}")